# Dev-set A/B: agent prompt v2 vs v3.2 (GPU L4 x4, internet OFF)
Same model (`gemma-4-31b-it-qat-w4a16-ct`), same tools (re-implemented: run_command, read_file, edit_file, write_file, get_status, submit_patch, graph tools), same limits (28 tool calls, 8-minute virtual clock calibrated to single-stream speed, 240 s per command) and sampling (temperature 0.2, top-p 0.95). Only the system prompt differs.

**Inputs:** competition data · model gemma-4-31b-it-qat-w4a16-ct · dataset `metric/gemma-4-developer-agent-wheelhouse` · optional: end-to-end output (to skip non-reproducing tasks) · **the earlier `ab_out.tgz` (seed 0) — required for the second-seed run**, so the output holds every run.

Output: `ab_out.tgz` → input of the grading notebook.

In [ ]:
%%writefile /kaggle/working/proplib.py
# proplib.py - Measured Propagator experiments: self-contained library (fresh notebook, CPU only).
# Needs only the competition data attached as notebook input. Everything else is built here.
import os, re, sys, json, time, shutil, subprocess, tarfile, ast, gc, tempfile, zipfile, tomllib
import xml.etree.ElementTree as ET
import concurrent.futures as cf
from collections import Counter, defaultdict
from email.parser import Parser
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as ssp
from packaging.requirements import Requirement
from packaging.specifiers import SpecifierSet
from packaging.utils import parse_wheel_filename, canonicalize_name
from packaging.tags import sys_tags
from packaging.markers import default_environment

# ============================== paths ==============================
WORKDIR = Path(os.environ.get('PROP_WORKDIR', '/kaggle/working'))
TMP = Path(os.environ.get('PROP_TMP', '/tmp/prop'))
OUT, ENVS = WORKDIR / 'propagator', WORKDIR / 'propenvs'
PLUG, SHIM, SHIM2 = WORKDIR / 'propplug', WORKDIR / 'propshim', WORKDIR / 'propshim_fallback'
SNAPS, WORK = TMP / 'snaps', TMP / 'work'
RD, TD = OUT / 'repro', OUT / 'tasks'
for _p in (OUT, ENVS, PLUG, SHIM, SHIM2, SNAPS, WORK, RD, TD): _p.mkdir(parents=True, exist_ok=True)
MEM_CAP_MB = int(os.environ.get('PROP_MEM_CAP_MB', 6000))
PY = sys.executable
TESTFILE = re.compile(r'(^|/)(tests?|testing)/.*\.py$|(^|/)test_[^/]*\.py$|_test\.py$')
NONPKG = {'tests', 'test', 'testing', 'docs', 'docs_src', 'scripts', 'examples', 'benchmarks', 'site', 'build', 'dist'}
PINS = {'fastapi': {'starlette': '==0.52.1', 'pydantic': '==2.13.4', 'pydantic-core': '==2.46.4'}}
TEST_EXTRA = ['pytest', 'pluggy', 'iniconfig', 'anyio']
REF_GATE = {'fastapi': dict(passed=2672, failed=1, error=102), 'rich': dict(passed=943, failed=9, error=0),
            'requests': dict(passed=414, failed=0, error=204), 'httpx': dict(passed=128, failed=0, error=0)}

def find_data():
    if os.environ.get('PROP_DATA'): return Path(os.environ['PROP_DATA'])
    base = Path('/kaggle/input')
    for pat in ('*/snapshots', '*/*/snapshots', '*/*/*/snapshots', '*/*/*/*/snapshots'):
        for d in sorted(base.glob(pat)):
            if d.is_dir() and any(d.glob('*.tgz')): return d.parent
    raise FileNotFoundError('Attach the competition data: no folder with snapshots/*.tgz under /kaggle/input')
DATA = find_data()

def load_tasks():
    for f in sorted(DATA.rglob('*.jsonl')):
        with open(f) as fh: first = fh.readline()
        if '"instance_id"' in first and '"patch"' in first:
            return [json.loads(l) for l in open(f) if l.strip()]
    raise FileNotFoundError('no tasks .jsonl with instance_id/patch under ' + str(DATA))
TASKS = load_tasks(); T = {t['instance_id']: t for t in TASKS}
repo_of = lambda t: t['repo'].split('/')[-1]
REPOS = list(dict.fromkeys(repo_of(t) for t in TASKS))
PILOT = {r: next(t for t in TASKS if repo_of(t) == r) for r in REPOS}

def find_wheels():
    c = Counter(p.parent for p in DATA.rglob('*.whl'))
    return c.most_common(1)[0][0] if c else None
WHEELS = find_wheels()

# ============================== snapshots ==============================
def extract_lite(task):
    dst = SNAPS / task['instance_id']
    if (dst / '.prop_done').exists(): return dst
    shutil.rmtree(dst, ignore_errors=True)
    with tarfile.open(DATA / 'snapshots' / f"{task['instance_id']}.tgz") as tf:
        keep = [m for m in tf.getmembers() if (m.isfile() or m.isdir())
                and (m.name[2:] if m.name.startswith('./') else m.name).split('/', 1)[0] != '.git']
        tf.extractall(dst, members=keep, filter='data')
    kids = [p for p in dst.iterdir()]
    if len(kids) == 1 and kids[0].is_dir() and not (dst / 'pyproject.toml').exists() and not (dst / 'setup.py').exists():
        inner = kids[0]                                   # tarball with a single top folder: flatten it
        for p in inner.iterdir(): shutil.move(str(p), dst / p.name)
        inner.rmdir()
    (dst / '.prop_done').write_text('1')
    return dst

def fresh_copy(task, name=None):
    w = WORK / (name or task['instance_id']); shutil.rmtree(w, ignore_errors=True)
    shutil.copytree(extract_lite(task), w, symlinks=True, ignore=shutil.ignore_patterns('.prop_done'))
    return w

def pkgs_of(root):
    root = Path(root)
    return sorted({d.name for b in (root, root / 'src') if b.is_dir() for d in b.iterdir()
                   if d.is_dir() and (d / '__init__.py').exists() and d.name not in NONPKG})

# ============================== processes (memory watchdog) ==============================
def _rss_mb(pid):
    try:
        with open(f'/proc/{pid}/status') as fh:
            for l in fh:
                if l.startswith('VmRSS:'): return int(l.split()[1]) // 1024
    except OSError:
        pass
    return 0

def sp(cmd, cwd, timeout, env=None, inp=None, cap_mb=None):
    """run a command; returns rc, output, secs, peak_rss_mb, killed_reason (None if not killed)"""
    cap = cap_mb or MEM_CAP_MB; t0 = time.time(); peak = 0; killed = None
    with tempfile.TemporaryFile('w+') as fo:
        p = subprocess.Popen([str(c) for c in cmd], cwd=str(cwd), env=env, text=True, stdout=fo, stderr=subprocess.STDOUT,
                             stdin=subprocess.PIPE if inp is not None else subprocess.DEVNULL, start_new_session=True)
        if inp is not None:
            try:
                p.stdin.write(inp); p.stdin.close()
            except BrokenPipeError:
                pass
        while p.poll() is None:
            r = _rss_mb(p.pid); peak = max(peak, r)
            if r > cap or time.time() - t0 > timeout:
                killed = f'rss>{cap}MB' if r > cap else 'timeout'
                try: os.killpg(p.pid, 9)
                except OSError: pass
            time.sleep(0.1)
        p.wait(); fo.seek(0); out = fo.read()
    return (124 if killed else p.returncode), out, time.time() - t0, peak, killed

# ============================== plugins and shims ==============================
STUB = '''import os
def pytest_addoption(parser):
    for name in os.environ.get("PROP_STUB_INI", "").split(","):
        if name: parser.addini(name, "stub: plugin unavailable offline", default="")
def pytest_configure(config):
    for m in os.environ.get("PROP_STUB_MARKERS", "").split(","):
        if m: config.addinivalue_line("markers", m + ": stub marker (plugin unavailable offline)")
'''
IDENT = ('def _ident(x=None, *a, **k):\n    return x\n'
         'snapshot = external = outsource = Is = _ident\n'
         'def __getattr__(name):\n    return _ident\n')
DIRTY = r'''
import re, uuid as _uuid, json as _json, copy as _copy
class DirtyEquals:
    def __init__(self, *a, **k): self._a, self._k = a, k
    def equals(self, o): return True
    def __eq__(self, o):
        try: return bool(self.equals(o))
        except Exception: return False
    def __ne__(self, o): return not self.__eq__(o)
    def __or__(self, o): return _Or(self, o)
    def __ror__(self, o): return _Or(o, self)
    def __and__(self, o): return _And(self, o)
    def __invert__(self): return _Not(self)
    __hash__ = object.__hash__
    def __repr__(self): return f'{type(self).__name__}{self._a!r}'
class _Or(DirtyEquals):
    def equals(self, o): return self._a[0] == o or self._a[1] == o
class _And(DirtyEquals):
    def equals(self, o): return self._a[0] == o and self._a[1] == o
class _Not(DirtyEquals):
    def equals(self, o): return not (self._a[0] == o)
class AnyThing(DirtyEquals): pass
class IsInstance(DirtyEquals):
    def equals(self, o): return isinstance(o, self._a[0])
class IsDict(DirtyEquals):
    _partial = False
    def __init__(self, *a, **k): super().__init__(*a, **k); self._d = dict(*a, **k)
    def settings(self, *, partial=None, strict=None, ignore=None):
        c = _copy.copy(self)
        if partial is not None: c._partial = partial
        return c
    def equals(self, o):
        if not isinstance(o, dict): return False
        if not self._partial and set(o) != set(self._d): return False
        return all(k in o and self._d[k] == o[k] for k in self._d)
class IsPartialDict(IsDict): _partial = True
class IsOneOf(DirtyEquals):
    def equals(self, o): return any(x == o for x in self._a)
class IsStr(DirtyEquals):
    def equals(self, o):
        r = self._k.get('regex'); return isinstance(o, str) and (r is None or re.fullmatch(r, o) is not None)
IsAnyStr = IsStr
class IsInt(DirtyEquals):
    def equals(self, o): return isinstance(o, int) and not isinstance(o, bool)
class IsFloat(DirtyEquals):
    def equals(self, o): return isinstance(o, float)
class IsList(DirtyEquals):
    def equals(self, o):
        if not isinstance(o, list) or ('length' in self._k and len(o) != self._k['length']): return False
        if not self._a: return True
        if self._k.get('check_order', True): return len(o) == len(self._a) and all(x == y for x, y in zip(self._a, o))
        rest = list(o)
        for x in self._a:
            for i, y in enumerate(rest):
                if x == y: rest.pop(i); break
            else: return False
        return not rest
class IsUUID(DirtyEquals):
    def equals(self, o):
        try: _uuid.UUID(str(o)); return True
        except Exception: return False
class HasRepr(DirtyEquals):
    def equals(self, o): return repr(o) == self._a[0]
class IsJson(DirtyEquals):
    def equals(self, o): v = _json.loads(o); return (not self._a) or self._a[0] == v
def __getattr__(name): raise ImportError(f'dirty_equals shim lacks {name}')
'''
TRACER = r'''
import os, sys, json
import pytest
_mon = sys.monitoring
_ROOT = os.path.realpath(os.environ.get("PROP_TRACE_ROOT", os.getcwd())) + os.sep
_PKGS = set(filter(None, os.environ.get("PROP_TRACE_PKG", "").split(",")))
_DIR = os.environ.get("PROP_TRACE_DIR", ".")
_TOOL = next(i for i in (2, 3, 4, 5) if _mon.get_tool(i) is None)
_ids, _nid, _edges, _collect, _fcache = {}, {}, set(), set(), {}
_cur = _collect
_fh = open(os.path.join(_DIR, ".trace_tests.jsonl"), "w")

def _modparts(fname):
    r = _fcache.get(fname, 0)
    if r != 0:
        return r
    r = None
    if fname.endswith(".py"):
        rf = os.path.realpath(fname)
        if rf.startswith(_ROOT):
            parts = rf[len(_ROOT):-3].split(os.sep)
            if parts and parts[0] == "src": parts = parts[1:]
            if parts and parts[-1] == "__init__": parts = parts[:-1]
            if parts and parts[0] in _PKGS: r = ".".join(parts)
    _fcache[fname] = r
    return r

def _start(code, offset):
    i = _ids.get(code)
    if i is None:
        mod = _modparts(code.co_filename)
        if mod is None:
            return _mon.DISABLE
        q = code.co_qualname.replace("<locals>.", "")
        q = "<module>" if q.startswith("<") else q.split(".<")[0]
        i = _ids[code] = _nid.setdefault(mod + "." + q, len(_nid))
    _cur.add(i)
    f = sys._getframe(1).f_back
    for _ in range(6):
        if f is None: break
        j = _ids.get(f.f_code)
        if j is not None:
            if j != i: _edges.add((j, i))
            break
        f = f.f_back

@pytest.hookimpl(hookwrapper=True)
def pytest_runtest_protocol(item, nextitem):
    global _cur
    _cur = set()
    try:
        yield
    finally:
        _fh.write(json.dumps([item.nodeid, sorted(_cur)]) + "\n")
        _cur = _collect

def pytest_sessionfinish(session, exitstatus):
    _mon.set_events(_TOOL, 0)
    _fh.close()
    nodes = [None] * len(_nid)
    for n, i in _nid.items(): nodes[i] = n
    with open(os.path.join(_DIR, ".trace_meta.json"), "w") as fh:
        json.dump({"nodes": nodes, "collect": sorted(_collect), "edges": sorted(_edges)}, fh)

_mon.use_tool_id(_TOOL, "prop_trace2")
_mon.register_callback(_TOOL, _mon.events.PY_START, _start)
_mon.set_events(_TOOL, _mon.events.PY_START)
'''
(PLUG / 'prop_stub.py').write_text(STUB); (PLUG / 'prop_trace2.py').write_text(TRACER)
(SHIM / 'inline_snapshot').mkdir(exist_ok=True)
(SHIM / 'inline_snapshot' / '__init__.py').write_text(IDENT); (SHIM / 'inline_snapshot' / 'extra.py').write_text(IDENT)

# ============================== per-repo environments (pip --target from wheels/) ==============================
def _wheel_index():
    supported = {str(t) for t in sys_tags()}; idx = defaultdict(list)
    for f in (WHEELS.glob('*.whl') if WHEELS else []):
        try:
            name, ver, _, tags = parse_wheel_filename(f.name)
        except Exception:
            continue
        if any(str(t) in supported for t in tags): idx[canonicalize_name(name)].append((ver, f))
    for k in idx: idx[k].sort(key=lambda x: x[0], reverse=True)
    return idx

def _wheel_requires(f):
    with zipfile.ZipFile(f) as z:
        meta = next((n for n in z.namelist() if n.endswith('.dist-info/METADATA')), None)
        if not meta: return []
        return Parser().parsestr(z.read(meta).decode('utf-8', 'replace')).get_all('Requires-Dist') or []

def _setup_py_reqs(path):
    out = []
    try:
        tree = ast.parse(path.read_text(errors='replace'))
    except SyntaxError:
        return out
    def strs(node):
        if isinstance(node, (ast.List, ast.Tuple)): return [e.value for e in node.elts if isinstance(e, ast.Constant) and isinstance(e.value, str)]
        if isinstance(node, ast.Dict): return [s for v in node.values for s in strs(v)]
        return []
    for n in ast.walk(tree):
        if isinstance(n, ast.Assign) and any(isinstance(t, ast.Name) and 'require' in t.id.lower() for t in n.targets): out += strs(n.value)
        if isinstance(n, ast.keyword) and n.arg and 'require' in n.arg: out += strs(n.value)
    return out

def repo_requirements(snap):
    snap = Path(snap); raw = []
    pp = snap / 'pyproject.toml'
    if pp.exists():
        d = tomllib.loads(pp.read_text(errors='replace')); proj = d.get('project', {})
        raw += proj.get('dependencies', [])
        for v in proj.get('optional-dependencies', {}).values(): raw += v
        groups = d.get('dependency-groups', {})
        def expand(g, seen=()):
            res = []
            for x in groups.get(g, []):
                if isinstance(x, str): res.append(x)
                elif isinstance(x, dict) and x.get('include-group') not in seen + (g,): res += expand(x['include-group'], seen + (g,))
            return res
        for g in groups: raw += expand(g)
        poetry = d.get('tool', {}).get('poetry', {})
        names = list(poetry.get('dependencies', {})) + list(poetry.get('dev-dependencies', {}))
        for g in poetry.get('group', {}).values(): names += list(g.get('dependencies', {}))
        raw += [n for n in names if n.lower() != 'python']
    for f in list(snap.glob('requirements*.txt')) + list((snap / 'requirements').glob('*.txt')):
        for line in f.read_text(errors='replace').splitlines():
            line = line.split('#')[0].strip()
            if line and not line.startswith(('-', 'git+', 'http', '.')): raw.append(line)
    if (snap / 'setup.py').exists(): raw += _setup_py_reqs(snap / 'setup.py')
    if (snap / 'setup.cfg').exists():
        txt = (snap / 'setup.cfg').read_text(errors='replace')
        for m in re.finditer(r'^(?:install_requires|tests_require)\s*=\s*\n((?:[ \t]+\S.*\n?)+)', txt, flags=re.M):
            raw += [l.strip() for l in m.group(1).splitlines() if l.strip()]
    reqs = []
    for s in raw + TEST_EXTRA:
        try: reqs.append(Requirement(s))
        except Exception: pass
    return reqs

def resolve(root_reqs, pins, skip):
    idx, env = _wheel_index(), default_environment(); chosen, missing, notes = {}, [], []
    ok = lambda r, ex: r.marker is None or any(r.marker.evaluate({**env, 'extra': e}) for e in (ex or {''}))
    queue = [(r, None) for r in root_reqs]
    while queue:
        r, pex = queue.pop(0)
        if not ok(r, pex): continue
        n = canonicalize_name(r.name)
        if n in skip: continue
        if n in chosen:
            v, f, ex = chosen[n]; new = set(r.extras) - ex
            if new:
                chosen[n] = (v, f, ex | new)
                queue += [(Requirement(d), new) for d in _wheel_requires(f)]
            continue
        spec = SpecifierSet(pins[n]) if n in pins else r.specifier
        cand = [(v, f) for v, f in idx.get(n, []) if spec.contains(v, prereleases=True)]
        if not cand and n in pins:
            notes.append(f'pin {n}{pins[n]} unavailable, using {r.specifier or "any"}')
            cand = [(v, f) for v, f in idx.get(n, []) if r.specifier.contains(v, prereleases=True)]
        if not cand: missing.append(str(r)); continue
        v, f = cand[0]; chosen[n] = (v, f, set(r.extras))
        queue += [(Requirement(d), set(r.extras) or {''}) for d in _wheel_requires(f)]
    return chosen, sorted(set(missing)), notes

def env_dir(repo): return ENVS / repo

def build_env(repo, force=False):
    d = env_dir(repo); done = d / '.prop_env.json'
    if done.exists() and not force: return json.load(open(done))
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    snap = extract_lite(PILOT[repo]); skip = {canonicalize_name(x) for x in pkgs_of(snap) + [repo]}
    chosen, missing, notes = resolve(repo_requirements(snap), {canonicalize_name(k): v for k, v in PINS.get(repo, {}).items()}, skip)
    files = [str(f) for _, f, _ in chosen.values()]; rc, out = 0, ''
    if files:
        rc, out, *_ = sp([PY, '-m', 'pip', 'install', '-q', '--no-deps', '--no-index', '--disable-pip-version-check',
                          '--no-compile', '--target', d] + files, '/tmp', 1800, cap_mb=20000)
    info = dict(repo=repo, installed={k: str(v) for k, (v, _, _) in chosen.items()}, missing=missing, notes=notes,
                pip_rc=rc, pip_tail=out[-600:])
    json.dump(info, open(done, 'w'), indent=1)
    return info

# ============================== running pytest ==============================
CFG_FILE = OUT / 'cfg.json'
CFG = json.load(open(CFG_FILE)) if CFG_FILE.exists() else {}

def env_for(repo, root, extra=None):
    root = Path(root); c = CFG.get(repo, {})
    env = {k: v for k, v in os.environ.items() if k not in ('COLUMNS', 'PYTHONPATH', 'MPLBACKEND', 'PYTHONSTARTUP')}
    pp = [str(root)] + ([str(root / 'src')] if (root / 'src').is_dir() else []) + [str(PLUG), str(SHIM), str(env_dir(repo)), str(SHIM2)]
    env.update(PYTHONPATH=os.pathsep.join(pp), PYTEST_DISABLE_PLUGIN_AUTOLOAD='1', PYTHONDONTWRITEBYTECODE='1',
               PROP_STUB_INI=','.join(c.get('ini_stubs') or []), PROP_STUB_MARKERS=','.join(c.get('markers') or []),
               OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
    env.update(extra or {})
    return env

def cmd_for(repo):
    c = CFG.get(repo, {})
    return ([PY, '-m', 'pytest', '-p', 'no:cacheprovider', '-q', '--continue-on-collection-errors', '-p', 'prop_stub']
            + (['-p', 'anyio.pytest_plugin'] if c.get('anyio') else []) + list(c.get('wflags') or []))

def junit_cases(xml):
    out = {}
    try:
        root = ET.parse(xml).getroot()
    except Exception:
        return out
    for tc in root.iter('testcase'):
        tags = {k.tag: k for k in tc}
        st = 'failed' if 'failure' in tags else 'error' if 'error' in tags else 'skipped' if 'skipped' in tags else 'passed'
        msg = ''
        for k in ('failure', 'error'):
            if k in tags:
                body = (tags[k].text or '').strip().splitlines()
                msg = ((tags[k].get('message') or '') + ' | ' + (body[-1] if body else ''))[:240]
        out[f"{tc.get('classname')}::{tc.get('name')}"] = (st, msg)
    return out

def suite_run(repo, w, tag, traced=False, timeout=1800, targets=()):
    w = Path(w); xml = w / f'.junit_{tag}.xml'; env = env_for(repo, w)
    cmd = cmd_for(repo) + list(targets) + [f'--junitxml={xml}']
    if traced:
        cmd += ['-p', 'prop_trace2']
        env.update(PROP_TRACE_DIR=str(w), PROP_TRACE_ROOT=str(w), PROP_TRACE_PKG=','.join(pkgs_of(w)))
    rc, out, secs, peak, killed = sp(cmd, w, timeout, env)
    return junit_cases(xml), dict(rc=rc, secs=round(secs, 1), peak_mb=peak, killed=killed, tail=out[-400:])

def detect_cfg(repo, force=False):
    """find ini options / markers of plugins we cannot load, and whether tests need the anyio plugin"""
    global CFG
    if repo in CFG and not force: return CFG[repo]
    w = fresh_copy(PILOT[repo], f'cfg_{repo}')
    tests_txt = ' '.join(p.read_text(errors='replace') for p in w.rglob('test*.py') if 'tests' in p.parts or p.name.startswith('test_'))
    c = dict(anyio=bool(re.search(r'pytest\.mark\.anyio|anyio_backend', tests_txt)), ini_stubs=[], markers=[], wflags=[])
    CFG[repo] = c
    for _ in range(6):
        rc, out, *_ = sp(cmd_for(repo) + ['--co'], w, 900, env_for(repo, w))
        ini = set(re.findall(r'Unknown config option: ([\w\-\.]+)', out)) - set(c['ini_stubs'])
        mk = set(re.findall(r"'([\w\.\-]+)' not found in `markers`", out)) - set(c['markers'])
        if not ini and not mk: break
        c['ini_stubs'] += sorted(ini); c['markers'] += sorted(mk)
    json.dump(CFG, open(CFG_FILE, 'w'), indent=1); shutil.rmtree(w, ignore_errors=True)
    return c

def ensure_dirty_equals():
    """fall back to a minimal dirty_equals shim only if no real one is importable"""
    for repo in REPOS:
        if sp([PY, '-c', 'import dirty_equals'], '/tmp', 120, env_for(repo, '/tmp'))[0] != 0:
            (SHIM2 / 'dirty_equals').mkdir(exist_ok=True); (SHIM2 / 'dirty_equals' / '__init__.py').write_text(DIRTY)
            return 'shim'
    return 'real'

def gate():
    rows = []
    for repo in REPOS:
        w = fresh_copy(PILOT[repo], f'gate_{repo}'); res, m = suite_run(repo, w, 'gate'); shutil.rmtree(w, ignore_errors=True)
        n = Counter(s for s, _ in res.values()); ref = REF_GATE.get(repo, {})
        rows.append(dict(repo=repo, pilot=PILOT[repo]['instance_id'], passed=n['passed'], failed=n['failed'], error=n['error'],
                         skipped=n['skipped'], ref=f"{ref.get('passed')}/{ref.get('failed')}/{ref.get('error')}",
                         secs=m['secs'], peak_mb=m['peak_mb'], killed=m['killed'],
                         ok=(n['passed'] >= 0.9 * ref['passed']) if ref else n['passed'] > 0))
    return pd.DataFrame(rows)

# ============================== patches ==============================
def apply_patch(patch, root):
    errs = []
    for label, p in (('raw', patch), ('nl', patch if patch.endswith('\n') else patch + '\n')):
        rc, out, *_ = sp(['git', 'apply', '--unsafe-paths', '-p1', '-'], root, 60, inp=p)
        if rc == 0: return True, f'{label}:git'
        errs.append(out.strip()[-160:])
        pc = ['patch', '-p1', '--batch', '--forward', '-s', '--fuzz=2', '--no-backup-if-mismatch']
        if sp(pc + ['--dry-run'], root, 60, inp=p)[0] == 0 and sp(pc, root, 60, inp=p)[0] == 0: return True, f'{label}:patch'
    return False, ' || '.join(errs[-2:])

def changed_lines(patch):
    files = defaultdict(set); cur = old = None
    for line in patch.replace('\r\n', '\n').split('\n'):
        if line.startswith('diff --git'):
            m = re.match(r'diff --git a/(\S+) b/(\S+)', line); cur = m.group(2) if m else None; old = None; continue
        if line.startswith('--- '):
            m = re.match(r'--- (?:a/)?(\S+)', line); path = m.group(1) if m else None
            cur = None if path in (None, '/dev/null') else path; old = None; continue
        if line.startswith('+++ '):
            m = re.match(r'\+\+\+ (?:b/)?(\S+)', line); path = m.group(1) if m else None
            if cur is None and path not in (None, '/dev/null'): cur = path
            continue
        if line.startswith('@@'):
            m = re.match(r'@@ -(\d+)', line); old = int(m.group(1)) if m else None; continue
        if cur is None or old is None or line.startswith('\\'): continue
        if line.startswith('-'): files[cur].add(old); old += 1
        elif line.startswith('+'): files[cur].update({max(old - 1, 1), old})
        else: old += 1
    return files

def _spans(src):
    out = []
    def walk(node, prefix):
        for ch in ast.iter_child_nodes(node):
            if isinstance(ch, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                q = prefix + ch.name
                out.append((min([ch.lineno] + [d.lineno for d in ch.decorator_list]), ch.end_lineno, q))
                walk(ch, q + '.')
            else:
                walk(ch, prefix)
    walk(ast.parse(src), '')
    return out

def modname(rel):
    parts = rel[:-3].split('/')
    if parts and parts[0] == 'src': parts = parts[1:]
    if parts and parts[-1] == '__init__': parts = parts[:-1]
    return '.'.join(parts)

def patch_functions(snap, patch):
    snap = Path(snap); pk = set(pkgs_of(snap)); F, notes = set(), []
    for rel, lines in changed_lines(patch).items():
        if not rel.endswith('.py') or TESTFILE.search(rel) or modname(rel).split('.')[0] not in pk:
            notes.append(f'skip {rel}'); continue
        if not (snap / rel).exists():
            notes.append(f'new file {rel}'); continue
        try:
            spans = _spans((snap / rel).read_text(errors='replace'))
        except SyntaxError:
            notes.append(f'syntax {rel}'); continue
        mod = modname(rel)
        for ln in lines:
            inside = [s for s in spans if s[0] <= ln <= s[1]]
            F.add(f'{mod}.{max(inside)[2]}' if inside else f'{mod}.<module>')
    return F, notes

def required_tests(t):
    for k in ('FAIL_TO_PASS', 'fail_to_pass'):
        v = t.get(k)
        if v:
            v = json.loads(v) if isinstance(v, str) else v
            return {x.split('::')[-1].split('[')[0] for x in v}, 'field'
    tp = t['test_patch']
    names = set(re.findall(r'^\+\s*(?:async\s+)?def\s+(test\w*)', tp, flags=re.M))
    names |= set(re.findall(r'^@@[^@]*@@.*?def\s+(test\w*)', tp, flags=re.M))
    return names, 'extracted'

def func_status(cases, req):
    st = defaultdict(list)
    for k, (s, _) in cases.items(): st[k.split('::', 1)[-1].split('[')[0]].append(s)
    return {n: ('missing' if not st.get(n) else 'pass' if all(s == 'passed' for s in st[n])
                else 'skip' if all(s == 'skipped' for s in st[n]) else 'fail') for n in req}

def test_targets(t, w):
    return sorted({f for f in re.findall(r'^\+\+\+ b/(\S+)', t['test_patch'], flags=re.M)
                   if TESTFILE.search(f) and not f.endswith('conftest.py') and (Path(w) / f).exists()})

# ============================== stage: gold-patch reproduction ==============================
def repro_task(t):
    tid, repo = t['instance_id'], repo_of(t); f = RD / f'{tid}.json'
    if f.exists(): return json.load(open(f))
    Fs, _ = patch_functions(extract_lite(t), t['patch']); Fp = [x for x in Fs if not x.endswith('.<module>')]
    res = dict(task=tid, repo=repo, cat='package' if Fp else 'module-level-only' if Fs else 'non-package', nF=len(Fp))
    w = fresh_copy(t)
    try:
        ok, how_t = apply_patch(t['test_patch'], w)
        if not ok: res.update(status='test_patch-failed', msg=how_t); return res
        targets = test_targets(t, w)
        if not targets: res.update(status='no-test-files'); return res
        req, kind = required_tests(t); runs = {}; how_p = ''
        for phase in ('base', 'gold'):
            if phase == 'gold':
                ok, how_p = apply_patch(t['patch'], w)
                if not ok: res.update(status='patch-failed', msg=how_p); return res
            runs[phase] = suite_run(repo, w, phase, timeout=600, targets=targets)
        cb, cg = runs['base'][0], runs['gold'][0]
        emp = sorted(k for k, (s, _) in cg.items() if s == 'passed' and cb.get(k, ('missing',))[0] in ('failed', 'error', 'missing'))
        sb, sg = func_status(cb, req), func_status(cg, req)
        f2p = sorted(n for n in req if sb[n] in ('fail', 'missing') and sg[n] == 'pass')
        bad = sorted(n for n in req if sg[n] != 'pass')
        status = (('reproduced-empirical' if emp else 'no-f2p') if not req else
                  'gold-fails' if bad else 'reproduced' if f2p else 'no-f2p')
        res.update(status=status, req_src=kind, n_req=len(req), f2p=f2p if req else emp, n_f2p=len(f2p) if req else len(emp),
                   apply=f'{how_t}/{how_p}', targets=len(targets), bad=[f'{n}:{sg[n]}' for n in bad][:4],
                   bad_msg=[m for k, (s, m) in cg.items() if s in ('failed', 'error')][:2] if bad else [],
                   secs=runs['base'][1]['secs'] + runs['gold'][1]['secs'])
        return res
    except Exception as e:
        res.update(status='crash', msg=repr(e)[:300]); return res
    finally:
        if res.get('status') != 'crash': json.dump(res, open(f, 'w'))
        shutil.rmtree(w, ignore_errors=True)

def run_repro(workers=4, log_every=15):
    rows, t0 = [], time.time()
    with cf.ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(repro_task, t): t for t in TASKS}
        for i, fu in enumerate(cf.as_completed(futs)):
            try:
                rows.append(fu.result())
            except Exception as e:
                t = futs[fu]; rows.append(dict(task=t['instance_id'], repo=repo_of(t), cat='?', status='crash', msg=repr(e)[:300]))
            if (i + 1) % log_every == 0: print(f'   repro {i + 1}/{len(TASKS)}  {time.time() - t0:.0f}s', flush=True)
    rep = pd.DataFrame(rows).sort_values('task')
    rep['ok'] = rep.status.isin(['reproduced', 'reproduced-empirical'])
    rep.to_csv(OUT / 'repro.csv', index=False)
    return rep

# ============================== stage: traced coverage + real-bug response ==============================
def load_trace(w):
    w = Path(w); meta = json.load(open(w / '.trace_meta.json')); keys, R, Cc = [], [], []
    with open(w / '.trace_tests.jsonl') as fh:
        for i, line in enumerate(fh):
            nid, ids = json.loads(line); keys.append(nid2key(nid))
            if ids: R.append(np.full(len(ids), i, np.int32)); Cc.append(np.asarray(ids, np.int32))
    r = np.concatenate(R) if R else np.zeros(0, np.int32); c = np.concatenate(Cc) if Cc else np.zeros(0, np.int32)
    C = ssp.csr_matrix((np.ones(len(r), np.int8), (r, c)), shape=(len(keys), len(meta['nodes'])))
    C.data[:] = 1
    return keys, meta['nodes'], C, meta

def nid2key(nid):
    p = nid.split('::'); f = p[0][:-3].replace('/', '.') if p[0].endswith('.py') else p[0]
    return '.'.join([f] + p[1:-1]) + '::' + p[-1]

isbad = lambda s: s in ('failed', 'error', 'missing')

def task_run(t, f2p_list):
    repo, tid = repo_of(t), t['instance_id']; d = TD / tid; d.mkdir(exist_ok=True)
    if (d / 'summary.json').exists(): return json.load(open(d / 'summary.json'))
    w = fresh_copy(t); row = dict(task=tid, repo=repo)
    try:
        base, m1 = suite_run(repo, w, 'base')
        if not apply_patch(t['patch'], w)[0]: return {**row, 'error': 'patch'}
        gold, m2 = suite_run(repo, w, 'gold')
        if not apply_patch(t['test_patch'], w)[0]: return {**row, 'error': 'test_patch'}
        tr, m3 = suite_run(repo, w, 'goldtp', traced=True)
        row.update(secs=[m1['secs'], m2['secs'], m3['secs']], peak_mb=[m1['peak_mb'], m2['peak_mb'], m3['peak_mb']])
        if any(m['killed'] for m in (m1, m2, m3)):
            return {**row, 'error': 'killed ' + str([m['killed'] for m in (m1, m2, m3)])}
        if not (w / '.trace_meta.json').exists(): return {**row, 'error': 'no trace: ' + m3['tail']}
        keys, funcs, C, meta = load_trace(w); fi = {f: i for i, f in enumerate(funcs)}
        d_old = sorted(k for k, (s, _) in gold.items() if s == 'passed' and isbad(base.get(k, ('missing',))[0]))
        d_rev = sorted(k for k, (s, _) in base.items() if s == 'passed' and isbad(gold.get(k, ('missing',))[0]))
        tpf = [f for f in re.findall(r'^\+\+\+ b/(\S+)', t['test_patch'], flags=re.M) if TESTFILE.search(f)]
        tmod = tuple(f[:-3].replace('/', '.') for f in tpf)
        in_t = lambda k: any(k.startswith(m + '.') or k.startswith(m + '::') or k == '::' + m for m in tmod)
        mod_names = required_tests(t)[0]                       # test functions the test patch adds or edits
        is_mod = lambda k: k.split('::')[-1].split('[')[0] in mod_names
        old_keys = {k for k in gold if not is_mod(k)}          # pre-existing tests (run without the test patch)
        mism = sorted(k for k in old_keys if tr.get(k, ('missing',))[0] != gold[k][0])
        ssp.save_npz(d / 'C.npz', C)
        json.dump({'tests': keys, 'funcs': funcs, 'collect': meta['collect'], 'edges': meta['edges']}, open(d / 'index.json', 'w'))
        Fs = sorted(patch_functions(extract_lite(t), t['patch'])[0]); Fp = [f for f in Fs if not f.endswith('.<module>')]
        Fcols = [fi[f] for f in Fp if f in fi]; lst = f2p_list or []
        f2p_keys = ({k for k in lst if '::' in k} if any('::' in x for x in lst) else
                    {k for k in keys if in_t(k) and k.split('::')[-1].split('[')[0] in set(lst)})
        rows_f2p = [i for i, k in enumerate(keys) if k in f2p_keys]
        old_rows = np.array([k in old_keys for k in keys], bool)
        hitsF = (np.asarray(C[:, Fcols].sum(1)).ravel() > 0) if Fcols else np.zeros(len(keys), bool)
        foot = int((np.asarray(C[rows_f2p].sum(0)).ravel() > 0).sum()) if rows_f2p else 0
        row.update(n_tests=len(keys), n_funcs=len(funcs), n_edges=len(meta['edges']), pkgs=','.join(pkgs_of(w)),
                   d_old=len(d_old), d_old_list=d_old[:8], d_rev=len(d_rev), d_rev_list=d_rev[:5],
                   fid_mismatch=len(mism), fid_list=mism[:5], nF=len(Fp), F_exec=len(Fcols),
                   F_missing=[f for f in Fp if f not in fi][:4], old_tests_hit_F=int((hitsF & old_rows).sum()),
                   n_f2p=len(rows_f2p), f2p_hit_F=bool(rows_f2p) and bool(hitsF[rows_f2p].any()), f2p_footprint=foot)
        json.dump(row, open(d / 'summary.json', 'w'))
        return row
    except Exception as e:
        return {**row, 'error': repr(e)[:300]}
    finally:
        shutil.rmtree(w, ignore_errors=True); gc.collect()

def run_step8(rep, log_every=10):
    use = rep[rep.ok & (rep['cat'] == 'package')]
    f2p = {x: json.load(open(RD / f'{x}.json')).get('f2p', []) for x in use.task}
    done = {p.parent.name for p in TD.glob('*/summary.json')}
    todo = [T[x] for x in use.task]
    probe = next((t for t in todo if t['instance_id'] not in done), None); rows = []
    workers = 2
    if probe is not None:
        r = task_run(probe, f2p[probe['instance_id']]); rows.append(r)
        peak = max(r.get('peak_mb') or [9999]); workers = 3 if peak < 2500 else 2
        print(f"   probe {probe['instance_id']}: secs {r.get('secs')} peak MB {r.get('peak_mb')} error {r.get('error')} -> workers {workers}", flush=True)
    order = {'requests': 0, 'httpx': 1, 'fastapi': 2, 'rich': 3}
    rest = sorted([t for t in todo if probe is None or t['instance_id'] != probe['instance_id']], key=lambda t: order.get(repo_of(t), 9))
    t0 = time.time()
    with cf.ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(task_run, t, f2p[t['instance_id']]): t for t in rest}
        for i, fu in enumerate(cf.as_completed(futs)):
            try:
                rows.append(fu.result())
            except Exception as e:
                t = futs[fu]; rows.append(dict(task=t['instance_id'], repo=repo_of(t), error=repr(e)[:300]))
            if (i + 1) % log_every == 0: print(f'   step8 {i + 1}/{len(rest)}  {time.time() - t0:.0f}s', flush=True)
    S = pd.DataFrame(rows).sort_values('task'); S.to_csv(OUT / 'step8_tasks.csv', index=False)
    return S

def report8(S):
    err = S[S['error'].notna()] if 'error' in S else S.iloc[:0]
    print(f'step 8: {len(S)} tasks, errors: {len(err)}')
    for _, r in err.iterrows(): print('   ERR', r.task, str(r.error)[:200])
    S = S[S['error'].isna()] if 'error' in S else S
    if S.empty: return
    print('\nper repo (median):')
    print(S.groupby('repo')[['n_tests', 'n_funcs', 'n_edges', 'nF', 'F_exec', 'n_f2p', 'f2p_footprint']].median().to_string())
    print('mean secs base/gold/traced:', {r: np.round(np.mean(g.secs.tolist(), axis=0), 1).tolist() for r, g in S.groupby('repo')})
    print('max peak MB base/gold/traced:', {r: np.max(g.peak_mb.tolist(), axis=0).tolist() for r, g in S.groupby('repo')})
    print(f"\nFIDELITY  tasks with 0 mismatches: {(S.fid_mismatch == 0).sum()}/{len(S)} | total {S.fid_mismatch.sum()}")
    for _, r in S[S.fid_mismatch > 0].nlargest(5, 'fid_mismatch').iterrows(): print(f'   {r.task}: {r.fid_mismatch} {r.fid_list[:3]}')
    print(f"\nNAMES  F* found among executed functions: {S.F_exec.sum()}/{S.nF.sum()} | "
          f"tasks whose F2P tests execute F*: {S.f2p_hit_F.sum()}/{(S.n_f2p > 0).sum()}")
    print('   F* names not seen (sample):', [x for l in S.F_missing for x in l][:8])
    print('\nREAL-BUG RESPONSE on the pre-existing suite (d_old = old tests failing on base, passing with fix):')
    print(f"   tasks with d_old = 0: {(S.d_old == 0).sum()}/{len(S)} | size: {S.d_old.describe()[['mean', '50%', 'max']].to_dict()}")
    print(f"   reverse flips (fix breaks an old test): {(S.d_rev > 0).sum()} tasks")
    cov = S[S.old_tests_hit_F > 0]
    print(f"   old suite EXECUTES F*: {len(cov)}/{len(S)} tasks (median {cov.old_tests_hit_F.median():.0f} old tests touch F*)")
    print(f"   ... and is SILENT (d_old = 0): {(cov.d_old == 0).sum()}/{len(cov)}")
    for _, r in S[S.d_old > 0].iterrows(): print(f'   d_old>0 {r.task}: {r.d_old} {r.d_old_list[:3]}')


In [ ]:
%%writefile /kaggle/working/ab_agent.py
# ab_agent.py - dev-set A/B of the competition agent's system prompt: v2 (as scored, 0.18) vs v3.1 (three habits).
# Same model (gemma-4-31b-it-qat-w4a16-ct), same tools, limits and sampling; only the system prompt differs.
# Needs proplib.py (cell above): environments are built offline from the competition wheels.
import os, re, sys, json, time, shutil, subprocess, urllib.request, urllib.error
import concurrent.futures as cf
from collections import Counter
from pathlib import Path
import pandas as pd
sys.path.insert(0, os.environ.get('PROP_WORKDIR', '/kaggle/working'))
import proplib as P

AB_OUT = P.WORKDIR / 'ab_out'; AB_WORK = Path(os.environ.get('AB_WORK', '/tmp/ab_work'))
for _p in (AB_OUT, AB_WORK): _p.mkdir(parents=True, exist_ok=True)
PROMPTS = json.loads(r'''{"v2": "You are an autonomous software engineer. Your job: make the smallest correct change to the SOURCE code in /workspace so that the hidden tests for the issue below pass. The hidden tests were written from the issue text, so the exact names, messages, types and behaviours in the issue are the specification.\n\n# HARD LIMITS\n- 28 tool calls and 8 minutes in total. When either runs out, the working tree is graded as it is. An edit that exists counts, even if you never call `submit_patch`. An unedited tree always scores zero.\n- Never create, modify or delete test files (`test_*.py`, `*_test.py`, anything under `tests/`). This is an automatic failure.\n- Never run a full test suite (bare `pytest`, `pytest .`, `unittest discover`). Always name one test file, and add `-k` when you can.\n- Never look outside /workspace. All dependencies are already installed.\n- Never use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`. They return nothing useful.\n\n# CALL PLAN\n- Calls 1-8: locate and read. Calls 9-12: edit. You MUST have edited a source file by call 12. If you are unsure, make your best-guess edit in the most likely place now. A wrong edit can still be corrected. Pure reading cannot score.\n- Calls 13-20: verify and correct.\n- Call 20 onward: no new exploration. Check, fix, submit.\n- Always finish with `git diff` followed by `submit_patch`.\n\n# PROCEDURE\n\n## 1. Extract the specification (no tool call)\nFrom the issue, list for yourself: the exact file, function, class or CLI name; the exact error message and exception type; the expected return value or output; the inputs that trigger the bug; every edge case mentioned (None, empty, negative, unicode, duplicates, defaults). Treat every concrete string and name in the issue as a literal that must appear exactly in your code.\n\n## 2. Locate\n- Use exact paths from the issue directly. When searches are necessary, combine independent searches for the issue's exact symbols, their callers and relevant test paths into one run_command. Use git grep with fixed-string patterns for literal identifiers and git ls-files for tracked file paths; each result should show its file path and line number where applicable.\n- Batch only searches whose inputs are already known. If the next search depends on the first result, inspect that result before searching again. Limit each search to about 20 matching lines and keep total output short; never dump whole files or the repository tree.\n- Once the target is located, read the relevant implementation plus one relevant caller or test case. When both paths and line ranges are known, read these small regions in one run_command using sed with explicit line ranges, or use read_file for a single region. Aim for about 40 lines per region and at most three regions per command. Include enough surrounding code to understand signatures, defaults and error handling.\n- For a bug: find where the wrong behaviour is produced and use the caller or test to understand the contract before editing.\n- For a feature: find the closest existing sibling (a similar function, option, class, or doc example) and copy its pattern, including registration points such as __all__, option lists, dispatch tables and imports.\n- Reuse paths and test names already discovered. Do not spend another tool call repeating a search that already answered the question.\n- If the issue includes a code snippet that reproduces the problem, you may run it once with python -c (one call, never create a file) to see the real error.\n\n## 3. Edit\n- Use edit_file for changes to existing code. Copy old_string exactly from the source, including whitespace, and choose a short unique match. Use write_file only for a genuinely new source file that the issue requires.\n- If edit_file reports no match or multiple matches, read the exact source region and retry with a smaller unique snippet. Never repeat an identical failed edit. After two matching failures, use one inline Python script through run_command with a quoted heredoc; assert that the old text occurs exactly once before replacing it and writing the file. Preserve unrelated content and do not rewrite the file from memory.\n- Fix the cause, not the symptom. Do not special-case the example from the issue.\n- Change only what the issue requires. Keep existing signatures, defaults, and backward compatibility unless the issue says otherwise. Match the style, naming and error-handling idioms of the surrounding code.\n- Use the exact exception class and exact message text from the issue. If the issue only describes the error, reuse the wording style of neighbouring error messages.\n- Handle every edge case you listed in step 1.\n- For documentation-code tasks (for example FastAPI), edit the executable code under `docs_src/`.\n\n## 4. Verify (at most 2 calls)\n- Find the test file with `find tests -name \"*keyword*.py\"`, then run `timeout 150 python -m pytest tests/path/test_x.py -x -q -k keyword`. Use `python3 -m unittest tests.test_x` for unittest projects.\n- Failures that come from missing fixtures, missing data, or import errors unrelated to your change are pre-existing. Ignore them. Never repair tests or the environment.\n- If a test related to your change fails, read the traceback, fix the source once, and rerun once. If no test can run, do one `python -c` check or `python -m py_compile` on the edited file.\n\n## 5. Final check and submit\n- Run `git diff`. Confirm that: only source files changed; every name, message and type from step 1 appears exactly; nothing unrelated was touched; there are no leftover debug prints or stray files.\n- Call `submit_patch`. Confirm `patch_size > 0` and `files_changed > 0`. If either is 0, you have not edited anything. Edit now and submit again.\n- End with a single-sentence summary of the fix.\n\n# DECISION RULES\n- Ambiguous issue: choose the most literal reading of the issue text and act on it. Do not ask questions and do not stall.\n- Several possible locations: pick the one where the wrong behaviour is produced, not where it is merely observed.\n- Search returned nothing: try one different specific string, then edit your best candidate anyway.\n- Stuck after a failed fix: make a different, smaller change. Do not repeat the same edit.\n- Never conclude that nothing needs to change. Every task requires a non-empty source patch.\n", "v32": "You are an autonomous software engineer. Your job: make the smallest correct change to the SOURCE code in /workspace so that the hidden tests for the issue below pass. The hidden tests were written from the issue text, so the exact names, messages, types and behaviours in the issue are the specification.\n\n# HARD LIMITS\n- 28 tool calls and 8 minutes in total. When either runs out, the working tree is graded as it is. An edit that exists counts, even if you never call `submit_patch`. An unedited tree always scores zero.\n- Never create, modify or delete test files (`test_*.py`, `*_test.py`, anything under `tests/`). This is an automatic failure.\n- Never run a full test suite (bare `pytest`, `pytest .`, `unittest discover`). Always name one test file, and add `-k` when you can.\n- Never look outside /workspace (the one exception is the helper /tmp/probe.py described below). All dependencies are already installed.\n- Never use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`. They return nothing useful.\n\n# CALL PLAN\n- Calls 1-8: locate and read. Calls 9-12: edit. You MUST have edited a source file by call 12. If you are unsure, make your best-guess edit in the most likely place now. A wrong edit can still be corrected. Pure reading cannot score.\n- Calls 13-20: verify and correct.\n- Call 20 onward: no new exploration. Check, fix, submit.\n- Always finish with `git diff` followed by `submit_patch`.\n\n# PROCEDURE\n\n## 1. Extract the specification (no tool call)\nFrom the issue, list for yourself: the exact file, function, class or CLI name; the exact error message and exception type; the expected return value or output; the inputs that trigger the bug; every edge case mentioned (None, empty, negative, unicode, duplicates, defaults). Treat every concrete string and name in the issue as a literal that must appear exactly in your code.\n\n## 2. Locate\n- Use exact paths from the issue directly. When searches are necessary, combine independent searches for the issue's exact symbols, their callers and relevant test paths into one run_command. Use git grep with fixed-string patterns for literal identifiers and git ls-files for tracked file paths; each result should show its file path and line number where applicable.\n- Batch only searches whose inputs are already known. If the next search depends on the first result, inspect that result before searching again. Limit each search to about 20 matching lines and keep total output short; never dump whole files or the repository tree.\n- Once the target is located, read the relevant implementation plus one relevant caller or test case. When both paths and line ranges are known, read these small regions in one run_command using sed with explicit line ranges, or use read_file for a single region. Aim for about 40 lines per region and at most three regions per command. Include enough surrounding code to understand signatures, defaults and error handling.\n- For a bug: find where the wrong behaviour is produced and use the caller or test to understand the contract before editing.\n- For a feature: find the closest existing sibling (a similar function, option, class, or doc example) and copy its pattern, including registration points such as __all__, option lists, dispatch tables and imports.\n- Reuse paths and test names already discovered. Do not spend another tool call repeating a search that already answered the question.\n- If the issue includes a code snippet that reproduces the problem, run it ONCE under the probe (see THE PROBE) instead of plain python -c. It prints the real traceback and then the repository functions that the failing run executed, most suspicious first. Start reading at the top of that list: in our measurements the fix is usually in, or one call away from, a function the failing run executes.\n\n## 3. Edit\n- Use edit_file for changes to existing code. Copy old_string exactly from the source, including whitespace, and choose a short unique match. Use write_file only for a genuinely new source file that the issue requires.\n- If edit_file reports no match or multiple matches, read the exact source region and retry with a smaller unique snippet. Never repeat an identical failed edit. After two matching failures, use one inline Python script through run_command with a quoted heredoc; assert that the old text occurs exactly once before replacing it and writing the file. Preserve unrelated content and do not rewrite the file from memory.\n- Fix the cause, not the symptom. Do not special-case the example from the issue.\n- Change only what the issue requires. Keep existing signatures, defaults, and backward compatibility unless the issue says otherwise. Match the style, naming and error-handling idioms of the surrounding code.\n- Use the exact exception class and exact message text from the issue. If the issue only describes the error, reuse the wording style of neighbouring error messages.\n- Handle every edge case you listed in step 1.\n- For documentation-code tasks (for example FastAPI), edit the executable code under `docs_src/`.\n\n## 4. Verify (at most 2 calls, in this order)\n1. Check the NEW behaviour first: the hidden tests were written from the issue, and the pre-existing tests rarely exercise the new behaviour (more than half of wrong fixes still pass them). Run the issue's exact example, or a one-line `python -c` built from the issue's names, values and messages, and compare the output, return value, exception type and message with step 1. If it does not match, fix the source once and re-run once.\n2. Then always run one regression check (on the dev set, a failing old test that executes the edit meant the patch was wrong 96% of the time, and it flagged about 4 in 10 wrong patches): `timeout 150 python /tmp/probe.py --focus F tests/a.py < /dev/null 2>&1 | tail -30`, with F the edited function and tests/a.py the test file named after the edited module (or the first hit of `git grep -l -w F -- tests '*test*.py'`). It lists the tests that execute F and do not pass. If one fails: when it asserts the old behaviour that the issue explicitly asks to change, leave it (never edit tests); when it fails on missing fixtures, data or imports unrelated to your change, ignore it; otherwise fix the source once.\n- unittest-only projects: use `python3 -m unittest tests.test_x` for step 2.\n\n## 5. Final check and submit\n- Run `git diff`. Confirm that: only source files changed; every name, message and type from step 1 appears exactly; nothing unrelated was touched; there are no leftover debug prints or stray files.\n- Call `submit_patch`. Confirm `patch_size > 0` and `files_changed > 0`. If either is 0, you have not edited anything. Edit now and submit again.\n- End with a single-sentence summary of the fix.\n\n# THE PROBE\n/tmp/probe.py traces which repository functions run (Python 3.8+, stdlib only). Create it the FIRST time you need it, inside the same run_command as its first use, by copying this block exactly:\n\ncat > /tmp/probe.py <<'PROBE'\nimport sys, os, re, itertools, traceback, collections\nargs = sys.argv[1:]; FOCUS = args[args.index('--focus') + 1] if '--focus' in args else ''\nTESTS = [x for i, x in enumerate(args) if not x.startswith('--') and (i == 0 or args[i - 1] != '--focus')]\nROOT = os.getcwd() + os.sep\nSKIP = re.compile(r'(^|/)(tests?|testing)/|(^|/)test_[^/]*\\.py$|_test\\.py$|conftest\\.py$|-packages/')\nkeys, per, last, cur, tick = dict(), dict(), dict(), ['run'], itertools.count()\ndef prof(fr, ev, arg):\n    if ev != 'call': return\n    c = fr.f_code; k = keys.get(c, 0)\n    if k == 0:\n        f = c.co_filename; rel = f[len(ROOT):] if f.startswith(ROOT) else None\n        k = keys[c] = None if rel is None or SKIP.search(rel) or c.co_name.startswith('<') else (rel, c.co_firstlineno, getattr(c, 'co_qualname', c.co_name))\n    if k and cur[0] in per: per[cur[0]][1].add(k); last[k] = next(tick)\nif TESTS:\n    import pytest\n    class Probe:\n        @pytest.hookimpl(hookwrapper=True)\n        def pytest_runtest_call(self, item):\n            cur[0] = item.nodeid; per[item.nodeid] = ['?', set()]; sys.setprofile(prof)\n            try: yield\n            finally: sys.setprofile(None)\n        def pytest_runtest_logreport(self, report):\n            if report.nodeid in per and (report.when == 'call' or report.failed): per[report.nodeid][0] = report.outcome if report.when == 'call' else 'error'\n    pytest.main(['-q', '-p', 'no:cacheprovider', '--tb=line'] + TESTS, plugins=[Probe()])\nelse:\n    code = sys.stdin.read(); per['run'] = ['passed', set()]; sys.setprofile(prof)\n    try: exec(compile(code, '<issue>', 'exec'), dict(__name__='__main__'))\n    except BaseException: sys.setprofile(None); per['run'][0] = 'failed'; traceback.print_exc(limit=-4)\n    finally: sys.setprofile(None)\nnm = lambda k: '%s:%d %s' % k\nif FOCUS:\n    hit = dict((n, o) for n, (o, s) in per.items() if any(k[2] == FOCUS or k[2].endswith('.' + FOCUS) for k in s))\n    bad = [n for n, o in hit.items() if o != 'passed']\n    print('\\n=== %d of %d tests execute %s; %d of those do not pass' % (len(hit), len(per), FOCUS, len(bad)))\n    for n in bad + [n for n in hit if n not in bad][:10]: print('  %-7s %s' % (hit[n], n))\nelse:\n    fail = [s for o, s in per.values() if o != 'passed']; tot = collections.Counter(k for _, s in per.values() for k in s)\n    ef = collections.Counter(k for s in fail for k in s); sc = dict((k, ef[k] / (len(fail) * tot[k]) ** 0.5) for k in ef)\n    print('\\n=== %d of %d runs fail; repo functions they execute, most suspicious first:' % (len(fail), len(per)) if fail else '\\n=== nothing failed; %d repo functions executed' % len(tot))\n    for k in (sorted(sc, key=lambda k: (-round(sc[k], 3), -last[k])) if fail else list(tot))[:25]: print('  %.2f %s' % (sc.get(k, 0), nm(k)))\nPROBE\n\nUsage (always from /workspace, always behind `timeout`):\n- Localize with the issue's snippet: `timeout 120 python /tmp/probe.py <<'EOF' 2>&1 | tail -40` then the snippet lines, then `EOF`.\n- Safety check after an edit: `timeout 150 python /tmp/probe.py --focus F tests/a.py < /dev/null 2>&1 | tail -30`.\nLater calls reuse the file: do not write it again. The probe never modifies /workspace.\n\n# DECISION RULES\n- Ambiguous issue: choose the most literal reading of the issue text and act on it. Do not ask questions and do not stall.\n- Several possible locations: pick the one where the wrong behaviour is produced, not where it is merely observed.\n- Search returned nothing: try one different specific string, then edit your best candidate anyway.\n- Stuck after a failed fix: make a different, smaller change. Do not repeat the same edit.\n- Never conclude that nothing needs to change. Every task requires a non-empty source patch.\n"}''')

# ============================== tools: a re-implementation of the competition agent's tool set ==============================
TOOL_LIMIT = 28; MAX_TURNS = 80; TIME_CAP_S = 480; CMD_TIMEOUT = 240; OUT_CHARS = (2000, 4000)
def _fn(name, desc, props, req):
    return {'type': 'function', 'function': {'name': name, 'description': desc,
            'parameters': {'type': 'object', 'properties': props, 'required': req}}}
S_ = lambda d: {'type': 'string', 'description': d}; I_ = lambda d: {'type': 'integer', 'description': d}
TOOLS = [
    _fn('run_command', 'Run a shell command in the repository (bash). Returns exit code and output.', {'command': S_('the command')}, ['command']),
    _fn('read_file', 'Read a file, optionally a line range (1-based, inclusive).', {'path': S_('file path'), 'start_line': I_('first line'), 'end_line': I_('last line')}, ['path']),
    _fn('edit_file', 'Replace exactly one occurrence of old_string with new_string in a file.', {'path': S_('file path'), 'old_string': S_('exact text to replace'), 'new_string': S_('replacement text')}, ['path', 'old_string', 'new_string']),
    _fn('write_file', 'Create or overwrite a file with the given content.', {'path': S_('file path'), 'content': S_('full file content')}, ['path', 'content']),
    _fn('get_status', 'Tool calls used and left, and elapsed time.', {}, []),
    _fn('submit_patch', 'Submit the current working tree as the patch.', {}, []),
    _fn('get_code_neighbors', 'Code graph neighbours of a symbol.', {'symbol': S_('symbol')}, ['symbol']),
    _fn('search_similar_code', 'Search similar code.', {'query': S_('query')}, ['query']),
    _fn('get_code_subgraph', 'Code subgraph around a symbol.', {'symbol': S_('symbol')}, ['symbol']),
]

def clip(s):
    a, b = OUT_CHARS
    return s if len(s) <= a + b else s[:a] + f'\n... [{len(s) - a - b} characters omitted] ...\n' + s[-b:]

class Workspace:
    def __init__(self, t, arm):
        self.t, self.repo = t, P.repo_of(t); base = AB_WORK / f"{arm}__{t['instance_id']}"
        shutil.rmtree(base, ignore_errors=True); self.w, self.tmp = base / 'ws', base / 'tmp'; self.tmp.mkdir(parents=True)
        shutil.copytree(P.extract_lite(t), self.w, symlinks=True, ignore=shutil.ignore_patterns('.prop_done'))
        for c in (['init', '-q'], ['add', '-A'], ['-c', 'user.email=a@b', '-c', 'user.name=ab', 'commit', '-qm', 'base', '--no-verify']):
            subprocess.run(['git'] + c, cwd=self.w, capture_output=True)
        c = P.CFG.get(self.repo, {}); addopts = ['-p', 'prop_stub', '-p', 'no:cacheprovider'] + (['-p', 'anyio.pytest_plugin'] if c.get('anyio') else [])
        self.env = P.env_for(self.repo, self.w, {'PYTEST_ADDOPTS': ' '.join(addopts + list(c.get('wflags') or [])), 'TMPDIR': str(self.tmp),
                                                 'PATH': str(Path(sys.executable).parent) + os.pathsep + os.environ.get('PATH', ''),
                                                 'GIT_PAGER': 'cat', 'PAGER': 'cat'})
    def path(self, p):
        q = Path(p); q = q if q.is_absolute() else self.w / q
        q = q.resolve()
        if not (str(q).startswith(str(self.w.resolve())) or str(q).startswith(str(self.tmp.resolve()))): raise ValueError(f'path outside the workspace: {p}')
        return q
    def call(self, name, a):
        if name == 'run_command':
            rc, out, secs, _, killed = P.sp(['bash', '-c', str(a.get('command', ''))], self.w, CMD_TIMEOUT, self.env, cap_mb=8000)
            return {'exit_code': rc, 'output': clip(out) + ('\n[killed: ' + killed + ']' if killed else '')}, secs
        if name == 'read_file':
            lines = self.path(a['path']).read_text(errors='replace').split('\n'); s = max(1, int(a.get('start_line') or 1))
            e = min(len(lines), int(a.get('end_line') or s + 399))
            return {'content': '\n'.join(f'{i}: {lines[i - 1]}' for i in range(s, e + 1)), 'total_lines': len(lines)}, 0.0
        if name == 'edit_file':
            p = self.path(a['path']); src = p.read_text(errors='replace'); n = src.count(a['old_string'])
            if n != 1: return {'error': f"old_string {'not found' if n == 0 else f'matches {n} times'} in {a['path']}"}, 0.0
            p.write_text(src.replace(a['old_string'], a['new_string'], 1)); return {'ok': True}, 0.0
        if name == 'write_file':
            p = self.path(a['path']); p.parent.mkdir(parents=True, exist_ok=True); p.write_text(a['content']); return {'ok': True}, 0.0
        if name == 'submit_patch':
            d = self.diff(); return {'patch_size': len(d), 'files_changed': len(re.findall(r'^diff --git', d, flags=re.M))}, 0.0
        if name in ('get_code_neighbors', 'search_similar_code', 'get_code_subgraph'): return {'results': []}, 0.0
        raise ValueError(f'unknown tool {name}')
    def diff(self):
        subprocess.run(['git', 'add', '-A'], cwd=self.w, capture_output=True)
        return subprocess.run(['git', 'diff', '--cached', '--no-color', '--', '.', ':(exclude).junit*'], cwd=self.w, capture_output=True, text=True).stdout

# ============================== model client (vLLM OpenAI-compatible server on localhost) ==============================
URL = 'http://127.0.0.1:8000/v1'; MODE = {'native': True}
def chat(messages, tools=True, max_tokens=4096, temperature=0.2, seed=0, think=False):
    body = {'model': 'gemma', 'messages': messages, 'max_tokens': max_tokens, 'temperature': temperature, 'top_p': 0.95, 'seed': seed,
            'chat_template_kwargs': {'enable_thinking': think}}
    if tools and MODE['native']: body.update(tools=TOOLS, tool_choice='auto')
    req = urllib.request.Request(URL + '/chat/completions', data=json.dumps(body).encode(), headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=1800) as r: return json.load(r)

TEXT_PROTOCOL = """

# TOOL CALLS
You act only through tools. To call one, reply with exactly one block and nothing after it:
```tool
{"name": "<tool name>", "arguments": {...}}
```
Tools: """ + json.dumps([{'name': x['function']['name'], 'parameters': x['function']['parameters']['properties']} for x in TOOLS])
def parse_text_calls(text):
    out = []
    for m in re.finditer(r'```(?:tool|json)?\s*(\{.*?\})\s*```', text or '', flags=re.S):
        try:
            j = json.loads(m.group(1))
            if 'name' in j: out.append({'id': f'c{len(out)}', 'function': {'name': j['name'], 'arguments': json.dumps(j.get('arguments') or j.get('args') or {})}})
        except Exception: pass
    return out[:1]


CTX, MAX_OUT, MIN_OUT = 32768, 4096, 1024
def est_tokens(msgs):                                    # conservative: ~3 characters per token for code and logs
    return sum(len(m.get('content') or '') + len(json.dumps(m.get('tool_calls') or '')) for m in msgs) // 3 + 200 * len(msgs) // 10
def compact(msgs, keep, hard=False):
    """shorten old tool results (all but the last `keep`), oldest first; True if something was shortened"""
    idx = [i for i, m in enumerate(msgs) if m['role'] in ('tool',) or (m['role'] == 'user' and m['content'].startswith('['))][:-keep or None]
    lim = 200 if hard else 600
    for i in idx:
        c = msgs[i]['content']
        if len(c) > lim + 60:
            msgs[i]['content'] = c[:lim // 2] + ' ...[older output truncated]... ' + c[-lim // 2:]; return True
    return False

def episode(t, arm, seed=0, think=False):
    tag = arm if seed == 0 else f'{arm}_s{seed}'                     # output folder per prompt and seed
    tid = t['instance_id']; f = AB_OUT / tag / f'{tid}.json'
    if f.exists(): return json.load(open(f))
    rec = dict(task=tid, arm=tag, prompt=arm, seed=seed, calls=0, turns=0, vtime=0.0, prompt_tokens=0, completion_tokens=0, tool_secs=0.0,
               submitted=False, stop='turns', errors=0, tools=Counter(), probe=False, compactions=0)
    try:
        ws = Workspace(t, tag)
        system = PROMPTS[arm].replace('/workspace', str(ws.w)).replace('/tmp/probe.py', str(ws.tmp / 'probe.py'))
        if not MODE['native']: system += TEXT_PROTOCOL
        user = f"Repository: {t.get('repo')} (checked out in {ws.w})\n\n<issue>\n{(t.get('problem_statement') or '').strip()}\n</issue>"
        msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]; prev, nudges = 0, 0
        for turn in range(MAX_TURNS):
            if rec['calls'] >= TOOL_LIMIT: rec['stop'] = 'tool_limit'; break
            if rec['vtime'] >= TIME_CAP_S: rec['stop'] = 'time_limit'; break
            r, err = None, None
            for attempt in range(3):                    # keep the context inside the window (same rule for both arms)
                keep = (6, 3, 1)[attempt]
                while est_tokens(msgs) + MIN_OUT > CTX and compact(msgs, keep): rec['compactions'] += 1
                max_tok = max(256, min(MAX_OUT, CTX - est_tokens(msgs) - 256))
                try: r = chat(msgs, seed=1000 * seed + turn, think=think, max_tokens=max_tok); break
                except urllib.error.HTTPError as e:
                    err = e.read().decode(errors='replace')[:400]
                    if e.code != 400 or 'context' not in err.lower(): break
                    compact(msgs, keep, hard=True); rec['compactions'] += 1
            if r is None:
                rec['stop'] = 'context_full' if err and 'context' in err.lower() else 'http_error'; rec['http_error'] = err; break
            rec['turns'] += 1; u = r.get('usage') or {}; pt, ct = u.get('prompt_tokens', 0), u.get('completion_tokens', 0)
            rec['prompt_tokens'] += pt; rec['completion_tokens'] += ct
            rec['vtime'] += max(pt - prev, 0) / RATES['prefill'] + ct / RATES['decode']; prev = pt + ct
            m = r['choices'][0]['message']; calls = m.get('tool_calls') or ([] if MODE['native'] else parse_text_calls(m.get('content')))
            amsg = {'role': 'assistant', 'content': m.get('content') or ''}
            if calls and MODE['native']: amsg['tool_calls'] = [{'id': c['id'], 'type': 'function', 'function': c['function']} for c in calls]
            msgs.append(amsg)
            if not calls:
                if nudges >= 2: rec['stop'] = 'no_tool_call'; break
                nudges += 1; msgs.append({'role': 'user', 'content': 'Continue the task using the tools. Finish with submit_patch.'}); continue
            done = False
            for c in calls:
                if rec['calls'] >= TOOL_LIMIT: break
                rec['calls'] += 1; name = c['function']['name']; rec['tools'][name] += 1
                try:
                    args = json.loads(c['function'].get('arguments') or '{}'); res, secs = ws.call(name, args)
                    if name == 'run_command' and 'probe.py' in str(args.get('command', '')): rec['probe'] = True
                except Exception as e:
                    res, secs = {'error': repr(e)[:300]}, 0.0; rec['errors'] += 1
                rec['tool_secs'] += secs; rec['vtime'] += secs
                content = json.dumps(res)
                msgs.append({'role': 'tool', 'tool_call_id': c['id'], 'content': content} if MODE['native'] else
                            {'role': 'user', 'content': f'[{name} result]\n{content}'})
                if name == 'submit_patch': rec['submitted'] = True; done = True
            if done: rec['stop'] = 'submitted'; break
        d = ws.diff(); rec['diff'] = d
        rec['touched_tests'] = any(P.TESTFILE.search(x) for x in re.findall(r'^diff --git a/(\S+)', d, flags=re.M))
        rec['transcript'] = [{'role': x['role'], 'content': (x.get('content') or '')[:600],
                              'tool_calls': [c['function'] for c in x.get('tool_calls', [])]} for x in msgs[2:]][:120]
    except Exception as e:
        rec['stop'] = 'crash'; rec['msg'] = repr(e)[:300]; rec.setdefault('diff', '')
        try: rec['diff'] = ws.diff()
        except Exception: pass
    rec['tools'] = dict(rec['tools']); (AB_OUT / tag).mkdir(parents=True, exist_ok=True); json.dump(rec, open(f, 'w'))
    shutil.rmtree(AB_WORK / f'{tag}__{tid}', ignore_errors=True)
    return rec

# ============================== server lifecycle and calibration ==============================
def start_server(model, tp, max_len=32768, util=0.85, max_seqs=32, kv_fp8=True, log=Path('/kaggle/working/vllm_server.log')):
    cmd = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server', '--model', model, '--served-model-name', 'gemma',
           '--tensor-parallel-size', str(tp), '--max-model-len', str(max_len), '--gpu-memory-utilization', str(util),
           '--max-num-seqs', str(max_seqs), '--enable-prefix-caching', '--enable-auto-tool-choice', '--tool-call-parser', 'gemma4',
           '--reasoning-parser', 'gemma4', '--port', '8000'] + (['--kv-cache-dtype', 'fp8'] if kv_fp8 else [])
    p = subprocess.Popen(cmd, stdout=open(log, 'w'), stderr=subprocess.STDOUT, start_new_session=True); t0 = time.time()
    while time.time() - t0 < 3600:
        if p.poll() is not None: raise RuntimeError('server exited:\n' + log.read_text()[-3000:])
        try:
            urllib.request.urlopen(URL + '/models', timeout=5); print(f'server up after {(time.time() - t0) / 60:.1f} min'); return p
        except Exception: time.sleep(10)
    raise RuntimeError('server did not start in 60 min')

def calibrate_and_probe_mode():
    """single-stream speeds (the scorer runs one task at a time) and whether native tool calls work"""
    r = chat([{'role': 'user', 'content': 'Use the run_command tool to run: echo hello'}], max_tokens=200)
    MODE['native'] = bool(r['choices'][0]['message'].get('tool_calls')); print('native tool calling:', MODE['native'], '|', str(r['choices'][0]['message'])[:200])
    t0 = time.time(); r = chat([{'role': 'user', 'content': 'Write a 300-word story about a lighthouse.'}], tools=False, max_tokens=400, temperature=0.0)
    RATES['decode'] = r['usage']['completion_tokens'] / (time.time() - t0)
    long = 'word ' * 6000; t0 = time.time(); r = chat([{'role': 'user', 'content': long + '\nSay OK.'}], tools=False, max_tokens=1, temperature=0.0)
    RATES['prefill'] = r['usage']['prompt_tokens'] / (time.time() - t0)
    print(f"single-stream decode {RATES['decode']:.1f} tok/s, prefill {RATES['prefill']:.0f} tok/s (used for the 8-minute virtual clock)")
RATES = {'decode': 20.0, 'prefill': 2000.0}

def run_ab(tasks, arms=('v2', 'v32'), workers=16, budget_s=9 * 3600, think=False, log_every=10, seed=0):
    jobs = [(t, a) for t in tasks for a in arms]; t0 = time.time(); rows = []
    def go(j):
        t, a = j
        tag = a if seed == 0 else f'{a}_s{seed}'
        if time.time() - t0 > budget_s and not (AB_OUT / tag / f"{t['instance_id']}.json").exists():
            return dict(task=t['instance_id'], arm=tag, stop='skipped-budget')
        return episode(t, a, seed=seed, think=think)
    with cf.ThreadPoolExecutor(workers) as ex:
        for k, r in enumerate(ex.map(go, jobs)):
            rows.append(r)
            if (k + 1) % log_every == 0:
                R = pd.DataFrame(rows); print(f"   {k + 1}/{len(jobs)} {(time.time() - t0) / 60:.0f} min | stop {R.stop.value_counts().to_dict()}", flush=True)
    return pd.DataFrame(rows)


In [ ]:
# ---- Dev-set A/B run (GPU L4 x4, internet OFF). Inputs: competition data (snapshots + wheels), the model
# google/gemma-4 -> gemma-4-31b-it-qat-w4a16-ct, the wheelhouse dataset (vLLM), and optionally the end-to-end output
# (propagator/repro/*.json) to skip the 18 tasks whose reference fix does not reproduce. Resumable: finished episodes are kept.
import os, sys, glob, json, time, subprocess, importlib
import pandas as pd
from pathlib import Path
sys.path.insert(0, '/kaggle/working')
def offline_vllm():
    """L4 sessions have no internet: install vLLM and its pinned runtime from the competition's wheel folder (no resolver, --no-deps)"""
    import re as _re
    whl = sorted(glob.glob('/kaggle/input/**/vllm-*.whl', recursive=True))
    if not whl:
        tops = sorted({'/'.join(p.split('/')[:5]) for p in glob.glob('/kaggle/input/*/*/*')})
        print('inputs attached:', *tops, sep='\n   ')
        raise RuntimeError('No vllm-*.whl under /kaggle/input. Add Input -> Datasets -> "gemma-4-developer-agent-wheelhouse" '
                           '(owner: metric). It holds vllm-0.19.1; the competition data alone does not.')
    folder = str(Path(whl[-1]).parent); files = sorted(glob.glob(folder + '/*.whl'))
    try:
        from packaging.version import Version as V
    except Exception:
        V = lambda x: x
    best = {}
    for f in files:                                   # one wheel per project (highest version)
        name, ver = Path(f).name.split('-')[:2]; key = _re.sub(r'[-_.]+', '-', name).lower()
        try: v = V(ver)
        except Exception: v = ver
        if key not in best or v > best[key][0]: best[key] = (v, f)
    picks = [f for _, f in best.values()]
    print(f'installing {len(picks)} wheels from {folder} with --no-deps (vLLM {Path(whl[-1]).name.split("-")[1]})')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '-q'] + picks, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-3000:])
    t = subprocess.run([sys.executable, '-c', 'import vllm, torch, google.protobuf as p; print("vllm", vllm.__version__, "torch", torch.__version__, "protobuf", p.__version__)'],
                       capture_output=True, text=True)
    print(t.stdout, t.stderr[-2500:])
try:
    import vllm
except ImportError:
    offline_vllm(); importlib.invalidate_caches(); import vllm
print('vLLM', vllm.__version__)
def find_model():
    if os.environ.get('GEMMA_PATH'): return os.environ['GEMMA_PATH']
    cands = []
    for cfg in glob.glob('/kaggle/input/**/config.json', recursive=True):
        try: c = json.load(open(cfg))
        except Exception: continue
        txt = json.dumps(c).lower(); d = str(Path(cfg).parent).lower()
        if 'gemma' in txt or 'gemma' in d: cands.append((('31b' in d) * 4 + ('w4a16' in d or 'compressed' in txt) * 2 + ('-it' in d), str(Path(cfg).parent)))
    if not cands: raise FileNotFoundError('add the model as an input: Kaggle Models -> google/gemma-4 -> gemma-4-31b-it-qat-w4a16-ct')
    return sorted(cands)[-1][1]
import ab_agent as A
for repo in A.P.REPOS:
    info = A.P.build_env(repo); print(f"env {repo:9s} {len(info['installed'])} wheels | cfg {A.P.detect_cfg(repo)}")
print('dirty_equals:', A.P.ensure_dirty_equals())
for tgz in glob.glob('/kaggle/input/**/ab_out.tgz', recursive=True)[:1]:          # resume from an attached earlier A/B run
    subprocess.run(f'cd /kaggle/working && tar xzf {tgz}', shell=True); print('restored', tgz)
ok = {json.load(open(f))['task'] for f in glob.glob('/kaggle/input/**/propagator/repro/*.json', recursive=True)
      if json.load(open(f)).get('status') == 'reproduced'}
TASKS = [A.P.T[i] for i in sorted(A.P.T) if not ok or i in ok]
print(f'{len(TASKS)} tasks x 2 arms' + ('' if ok else ' (no repro info attached: running all tasks)'))

MODEL = find_model(); print('model:', MODEL)
try:
    SRV = A.start_server(MODEL, tp=4, kv_fp8=True)
except RuntimeError as e:
    print('fp8 KV cache failed, retrying with the default KV cache:', str(e)[-400:])
    SRV = A.start_server(MODEL, tp=4, kv_fp8=False)
A.calibrate_and_probe_mode()
WORKERS  = int(os.environ.get('AB_WORKERS', 16))     # concurrent episodes sharing the server
BUDGET_S = 9 * 3600                                  # stop starting new episodes after 9 h; rerun with ab_out.tgz attached to continue
THINK    = False                                     # thinking off in both arms (cost); set True to mirror thinking_budget
SEEDS    = [1]                                       # seed 0 = the first run (ab_out/v2, ab_out/v32); [1, 2] adds two more runs (~4.6 h)
print('new seeds', SEEDS, '| earlier runs restored:', sorted(p.name for p in A.AB_OUT.iterdir()) if A.AB_OUT.exists() else [])
t0 = time.time()
R = pd.concat([A.run_ab(TASKS, arms=('v2', 'v32'), workers=WORKERS, budget_s=BUDGET_S, think=THINK, seed=s) for s in SEEDS])
print(f'done in {(time.time() - t0) / 3600:.1f} h'); print(R.groupby('arm').stop.value_counts().unstack(fill_value=0).to_string())
print(R.groupby('arm')[['calls', 'turns', 'vtime', 'completion_tokens', 'compactions']].mean().round(1).to_string())
for _, x in R[R.stop.isin(['crash', 'http_error', 'context_full'])].head(8).iterrows(): print('  ', x.arm, x.task, x.stop, str(x.get('msg') or x.get('http_error'))[:200])
subprocess.run('cd /kaggle/working && tar czf ab_out.tgz ab_out vllm_server.log && ls -la ab_out.tgz', shell=True)
